# Análisis Sísmico de Edificio de Concreto Armado con OpenSeesPy

Este notebook sigue un flujo de trabajo completo para el análisis de un edificio de varios pisos, similar a un tutorial estándar de SAP2000. Cubre:
1.  **Configuración del Proyecto y Datos**: Definición de la geometría, materiales y secciones.
2.  **Modelado**: Creación de nodos, columnas, vigas y diafragmas rígidos.
3.  **Cargas y Masas**: Asignación de cargas gravitacionales y cálculo de masas nodales.
4.  **Análisis Modal**: Cálculo de periodos y modos de vibración.
5.  **Análisis Sísmico Estático**: Aplicación de fuerzas laterales equivalentes y revisión de derivas.
6.  **Visualización de Resultados**: Gráficos de deformadas, modos y diagramas de fuerzas.

## 1. Configuración del Proyecto y Datos

In [ ]:
# Instalación de librerías (si es necesario)
!pip install openseespy opsvis pandas numpy matplotlib

In [ ]:
# Importación de librerías
import openseespy.opensees as ops
import opsvis as opsv
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import warnings

warnings.filterwarnings("ignore")
%matplotlib inline

print("Librerías importadas.")

In [ ]:
# ---- Unidades y Constantes ----
# Unidades Base
m = 1.0
kg = 1.0
s = 1.0

# Otras Unidades
cm = 0.01 * m
N = kg * m / s**2
kgf = 9.81 * N
tonf = 1000 * kgf

# Constantes Físicas
g = 9.81 * m / s**2

print(f"Unidades y constantes definidas. g = {g} m/s^2")

In [ ]:
# ---- Generación Programática de Geometría (Reemplaza a data.xlsx) ----

# Dimensiones de la cuadrícula
num_pisos = 3
num_crujias_x = 3 # Basado en losas, parece haber 4 ejes en X
num_crujias_y = 2 # Basado en losas, parece haber 3 ejes en Y
long_crujia_x = 4.0 # m, Asumido
long_crujia_y = 5.0 # m, Asumido
altura_piso = 3.0 # m

x_coords = np.arange(num_crujias_x + 1) * long_crujia_x
y_coords = np.arange(num_crujias_y + 1) * long_crujia_y
z_coords = np.arange(num_pisos + 1) * altura_piso

node_coords = []
for z in z_coords:
    for y in y_coords:
        for x in x_coords:
            node_coords.append([x, y, z])

nodos = np.array(node_coords)
num_nodos_total = len(nodos)
num_nodos_por_piso = (num_crujias_x + 1) * (num_crujias_y + 1)

print(f"Geometría generada: {num_pisos} pisos, {num_nodos_total} nodos en total.")
print(f"Dimensiones en X: {x_coords}")
print(f"Dimensiones en Y: {y_coords}")
print(f"Alturas en Z: {z_coords}")

In [ ]:
# ---- Generación Programática de Elementos ----
elementos_columna = []
for p in range(num_pisos):
    for i in range(num_nodos_por_piso):
        nodo_inf = p * num_nodos_por_piso + i + 1
        nodo_sup = (p + 1) * num_nodos_por_piso + i + 1
        elementos_columna.append([nodo_inf, nodo_sup])

elementos_viga = []
for p in range(1, num_pisos + 1): # Empezar desde el primer piso (z > 0)
    for j in range(len(y_coords)):
        for i in range(len(x_coords) - 1):
            # Vigas en dirección X
            n1 = p * num_nodos_por_piso + j * len(x_coords) + i + 1
            n2 = n1 + 1
            elementos_viga.append([n1, n2])
    for i in range(len(x_coords)):
        for j in range(len(y_coords) - 1):
            # Vigas en dirección Y
            n1 = p * num_nodos_por_piso + j * len(x_coords) + i + 1
            n2 = n1 + len(x_coords)
            elementos_viga.append([n1, n2])

print(f"Conectividad generada: {len(elementos_columna)} columnas, {len(elementos_viga)} vigas.")

In [ ]:
# ---- Propiedades de Material y Sección ----
fc = 280 * kgf / cm**2 # Pa
E = 15000 * np.sqrt(fc / (kgf/cm**2)) * (kgf/cm**2) # Ecuación ACI en Pa
G = 0.5 * E / (1 + 0.2) # Módulo de corte, asumiendo Poisson = 0.2 para concreto
rho = 2400 * kg / m**3 # Densidad del concreto

# Propiedades de Viga
b_viga, h_viga = 30*cm, 60*cm
A_viga = b_viga * h_viga
Iz_viga = b_viga * h_viga**3 / 12
Iy_viga = h_viga * b_viga**3 / 12
aa_v, bb_v = max(b_viga, h_viga), min(b_viga, h_viga)
beta_v = 1/3 - 0.21 * bb_v/aa_v * (1 - (bb_v/aa_v)**4 / 12)
J_viga = beta_v * bb_v**3 * aa_v

# Propiedades de Columna
a_col = 45*cm
A_col = a_col**2
Iz_col = a_col**4 / 12
Iy_col = a_col**4 / 12
beta_c = 1/3 - 0.21 * 1 * (1 - 1**4 / 12)
J_col = beta_c * a_col**4

print(f"Material Concreto: E = {E/1e9:.2f} GPa")
print(f"Sección Viga: {b_viga*100:.0f}x{h_viga*100:.0f} cm")
print(f"Sección Columna: {a_col*100:.0f}x{a_col*100:.0f} cm")

## 2. Creación del Modelo

In [ ]:
# Limpiar modelo anterior e inicializar uno nuevo
ops.wipe()
ops.model('basic', '-ndm', 3, '-ndf', 6)

print("Modelo inicializado.")

In [ ]:
# Creación de Nodos
for i, coords in enumerate(nodos):
    ops.node(i + 1, *coords)
print(f"{len(nodos)} nodos creados.")

In [ ]:
# Definición de Transformadas Geométricas
ops.geomTransf('PDelta', 1) # Para columnas (considera efectos P-Delta)
ops.geomTransf('Linear', 2, 0, 0, 1) # Para vigas
print("Transformaciones geométricas definidas: PDelta (1) y Linear (2).")

In [ ]:
# Creación de Elementos
ele_tag = 1

# Columnas
for nodos_col in elementos_columna:
    ops.element('elasticBeamColumn', ele_tag, *nodos_col, A_col, E, G, J_col, Iy_col, Iz_col, 1, '-mass', rho * A_col)
    ele_tag += 1

# Vigas
for nodos_viga in elementos_viga:
    ops.element('elasticBeamColumn', ele_tag, *nodos_viga, A_viga, E, G, J_viga, Iy_viga, Iz_viga, 2, '-mass', rho * A_viga)
    ele_tag += 1

print(f"{ele_tag - 1} elementos creados ({len(elementos_columna)} columnas y {len(elementos_viga)} vigas).")

In [ ]:
# Definición de Restricciones en la Base
base_nodes = np.arange(1, num_nodos_por_piso + 1)
for node in base_nodes:
    ops.fix(int(node), 1, 1, 1, 1, 1, 1) # Empotrado

print(f"Restricciones de empotramiento aplicadas a los {len(base_nodes)} nodos de la base.")

In [ ]:
# Visualización del Modelo de Alambre
plt.figure(figsize=(10, 12))
opsv.plot_model(show_nodes=False, show_ele_labels=False, title="Modelo de Alambre del Edificio")
plt.show()

## 3. Diafragmas y Cálculo de Masa

In [ ]:
# ---- Cálculo de Masa por Tributación de Área ----

# Cargas (en Pa = N/m^2)
wLosa = 300 * kgf / m**2
wAcab = 100 * kgf / m**2
wTabi = 150 * kgf / m**2
wLive = 250 * kgf / m**2

# Combinación de Carga para Masa Sísmica (1.0*CM + 0.25*CV)
w_total_mass = (wLosa + wAcab + wTabi) + 0.25 * wLive

# Función para calcular área tributaria (simplificada)
def get_tributary_area(node_index, nx, ny, dx, dy):
    ix = node_index % nx
    iy = node_index // nx
    
    if ix == 0 or ix == nx - 1: # Borde en X
        area_x = dx / 2
    else: # Interior en X
        area_x = dx
        
    if iy == 0 or iy == ny - 1: # Borde en Y
        area_y = dy / 2
    else: # Interior en Y
        area_y = dy
        
    return area_x * area_y

masses_per_floor_node = []
nx_grid = len(x_coords)
ny_grid = len(y_coords)

for i in range(num_nodos_por_piso):
    area_trib = get_tributary_area(i, nx_grid, ny_grid, long_crujia_x, long_crujia_y)
    nodal_mass = (area_trib * w_total_mass) / g # Masa = Peso / g
    masses_per_floor_node.append(nodal_mass)

# Asignar masas a todos los nodos de los pisos (excepto la base)
for p in range(1, num_pisos + 1):
    for i in range(num_nodos_por_piso):
        node_tag = p * num_nodos_por_piso + i + 1
        nodal_mass = masses_per_floor_node[i]
        ops.mass(node_tag, nodal_mass, nodal_mass, 0.0) # Masa en X e Y, no en Z

print(f"Masas nodales asignadas a {num_pisos * num_nodos_por_piso} nodos.")

In [ ]:
# ---- Creación de Nodos de Centro de Masa y Diafragmas Rígidos ----
cm_nodes = []
last_node = num_nodos_total

for p in range(1, num_pisos + 1):
    floor_nodes = np.arange( (p-1)*num_nodos_por_piso + 1, p*num_nodos_por_piso + 1)
    
    # Calcular CM (para una grilla regular, es el centro geométrico)
    x_cm = np.mean(x_coords)
    y_cm = np.mean(y_coords)
    z_cm = z_coords[p]
    
    # Crear nodo maestro en el CM
    cm_node_tag = last_node + p
    ops.node(cm_node_tag, x_cm, y_cm, z_cm)
    cm_nodes.append(cm_node_tag)
    
    # Asignar una masa muy pequeña al nodo maestro para evitar singularidades
    ops.mass(cm_node_tag, 1e-6, 1e-6, 1e-6)
    
    # Fijar grados de libertad que no son del plano del diafragma
    ops.fix(cm_node_tag, 0, 0, 1, 1, 1, 0) # Libre en UX, UY, RZ. Restringido en UZ, RX, RY.
    
    # Aplicar diafragma rígido
    ops.rigidDiaphragm(3, cm_node_tag, *floor_nodes.tolist()) # Diafragma en plano XY (perp a Z=3)

print(f"{num_pisos} diafragmas rígidos creados con nodos maestros: {cm_nodes}")

In [ ]:
# Visualización del Modelo con Diafragmas
plt.figure(figsize=(10, 12))
opsv.plot_model(show_nodes=True, show_ele_labels=False, title="Modelo con Nodos de Diafragma")
plt.show()

## 4. Análisis Modal

In [ ]:
# Cálculo de Modos de Vibración (Eigenvalores)
num_modes = 9
eigen_values = ops.eigen(num_modes)
omega = np.sqrt(np.array(eigen_values))
T_modes = (2 * np.pi) / omega

modal_results = pd.DataFrame({
    'Modo': range(1, num_modes + 1),
    'Eigenvalor (rad^2/s^2)': eigen_values,
    'Frecuencia (rad/s)': omega,
    'Periodo (s)': T_modes
})

print("Resultados del Análisis Modal:")
display(modal_results.round(4))

In [ ]:
# Visualización de Formas Modales
print("Visualizando las primeras 3 formas modales...")
scale_factor_modes = 5.0 # Ajustar para una buena visualización

for i in range(1, 4):
    plt.figure(figsize=(10, 12))
    opsv.plot_mode_shape(mode_no=i, scale_fact=scale_factor_modes)
    plt.title(f"Forma Modal {i} (T = {T_modes[i-1]:.3f} s)")
    plt.show()

In [ ]:
# Visualización del Modelo Extruido
ele_shapes = {}
ele_tag = 1
# Columnas
for _ in elementos_columna:
    ele_shapes[ele_tag] = ['rect', [a_col, a_col]]
    ele_tag += 1
# Vigas
for _ in elementos_viga:
    ele_shapes[ele_tag] = ['rect', [h_viga, b_viga]]
    ele_tag += 1

plt.figure(figsize=(12, 15))
opsv.plot_extruded_shapes_3d(ele_shapes)
plt.title('Modelo Extruido de la Estructura', fontweight='bold')
plt.show()

## 5. Análisis Sísmico Estático (Fuerzas Laterales Equivalentes)

In [ ]:
# ---- Parámetros y Función de Espectro de Respuesta ----

def espectro_E030(T, Z=0.45, U=1.0, S=1.0, Tp=0.4, Tl=2.5, R=8.0):
    """Calcula la pseudo-aceleración del espectro de la norma E.030 peruana."""
    C = np.zeros_like(T)
    for i, t_val in enumerate(T):
        if t_val < Tp:
            C[i] = 2.5
        elif t_val < Tl:
            C[i] = 2.5 * (Tp / t_val)
        else:
            C[i] = 2.5 * (Tp * Tl / t_val**2)
    Sa = (Z * U * C * S * g) / R
    return Sa

print("Función de espectro de respuesta definida.")

In [ ]:
# ---- Cálculo de Fuerzas Sísmicas Laterales ----

# Peso total por piso
total_mass_per_floor = sum(masses_per_floor_node)
W_p = total_mass_per_floor * g # Peso por piso
print(f"Peso total por piso (Wp): {W_p/tonf:.2f} tonf")

# Cortante Basal
T_fundamental = T_modes[0] # Periodo fundamental en la dirección principal (asumido X)
Sa_T1 = espectro_E030(np.array([T_fundamental]))[0]
V_basal = (Sa_T1 / g) * (W_p * num_pisos)
print(f"Cortante Basal (V): {V_basal/tonf:.2f} tonf")

# Distribución de fuerzas en altura
H = z_coords[1:] # Alturas de cada piso desde la base
k = 1.0 if T_fundamental <= 0.5 else (0.75 + 0.5 * T_fundamental) if T_fundamental < 2.5 else 2.0

Cv = (W_p * H**k) / np.sum(W_p * H**k)
F_lateral = Cv * V_basal

fuerzas_sismicas = pd.DataFrame({
    'Piso': range(1, num_pisos + 1),
    'Altura (m)': H,
    'Fuerza Lateral (tonf)': F_lateral / tonf
})
print("\nFuerzas Laterales Equivalentes (en X):")
display(fuerzas_sismicas)

In [ ]:
# ---- Aplicación de Cargas y Análisis Estático ----
ops.wipeAnalysis()
ops.setTime(0.0)

# Patrón de carga para sismo en X
ops.timeSeries('Linear', 3)
ops.pattern('Plain', 3, 3)

# Excentricidad accidental
e_acc = 0.05 * (long_crujia_x * num_crujias_x)

for i in range(num_pisos):
    cm_node = cm_nodes[i]
    Fx = F_lateral[i]
    Mz_torsion = Fx * e_acc # Momento torsional por excentricidad
    ops.load(cm_node, Fx, 0.0, 0.0, 0.0, 0.0, Mz_torsion)

print("Fuerzas sísmicas y momentos torsionales aplicados a los nodos CM.")

# Configuración del análisis
ops.system('BandGeneral')
ops.numberer('RCM')
ops.constraints('Transformation')
ops.integrator('LoadControl', 1.0)
ops.algorithm('Linear')
ops.analysis('Static')

# Ejecutar análisis
ok = ops.analyze(1)

if ok == 0:
    print("\nAnálisis estático sísmico completado exitosamente.")
else:
    print("\nError en el análisis estático sísmico.")

In [ ]:
# ---- Procesamiento y Visualización de Resultados (Derivas) ----
if ok == 0:
    resultados_derivas = pd.DataFrame(columns=['Piso', 'V_actuante(tonf)', 'Ux_max(cm)', 'Uy_max(cm)', 'Drift_X(‰)', 'Drift_Y(‰)'])
    
    # Parámetros para resultados
    R = 8.0 # Factor de reducción sísmica
    Cd = 0.75 * R # Factor de amplificación de desplazamientos (aproximado)
    story_shear = np.cumsum(F_lateral[::-1])[::-1]

    # Desplazamiento del piso anterior para cálculo de deriva
    ux_prev = 0
    uy_prev = 0

    for i in range(num_pisos):
        piso = i + 1
        cm_node = cm_nodes[i]
        
        # Desplazamiento elástico en el CM
        ux_elastic = ops.nodeDisp(cm_node, 1)
        uy_elastic = ops.nodeDisp(cm_node, 2)
        
        # Desplazamiento inelástico (amplificado)
        ux_inelastic = ux_elastic * Cd
        uy_inelastic = uy_elastic * Cd
        
        # Deriva de entrepiso (drift)
        drift_x = (ux_inelastic - ux_prev) / altura_piso
        drift_y = (uy_inelastic - uy_prev) / altura_piso
        
        resultados_derivas.loc[i] = [
            piso,
            story_shear[i] / tonf,
            ux_inelastic * 100, # a cm
            uy_inelastic * 100, # a cm
            drift_x * 1000, # a por mil
            drift_y * 1000  # a por mil
        ]
        
        # Actualizar desplazamientos para el siguiente piso
        ux_prev = ux_inelastic
        uy_prev = uy_inelastic
        
    print("Resultados de Desplazamientos y Derivas de Entrepiso (Sismo en X):")
    display(resultados_derivas.round(4))

    # Visualización de la deformada
    scale_defo = 50 # Ajustar para buena visualización
    plt.figure(figsize=(10, 12))
    opsv.plot_defo(scale_defo, az_el=(-60, 15))
    plt.title(f'Deformada por Sismo en X (Escala: {scale_defo})', fontweight='bold')
    plt.show()
else:
    print("No se pueden procesar resultados debido a error en el análisis.")

## 6. Visualización de Resultados (Diagramas de Fuerzas)

In [ ]:
if ok == 0:
    print("Generando diagramas de fuerzas para el caso de sismo estático en X...")

    # Diagrama de Fuerza Axial (N)
    sfacN = 5e-5 # Factor de escala para axiales
    plt.figure(figsize=(12, 15))
    opsv.section_force_diagram_3d('N', sfac=sfacN)
    plt.title('Diagrama de Fuerza Axial (N)', fontweight='bold')
    plt.show()

    # Diagrama de Fuerza Cortante (Vy)
    sfacV = 5e-4 # Factor de escala para cortantes
    plt.figure(figsize=(12, 15))
    opsv.section_force_diagram_3d('Vy', sfac=sfacV)
    plt.title('Diagrama de Fuerza Cortante (Vy)', fontweight='bold')
    plt.show()

    # Diagrama de Momento Flector (Mz)
    sfacM = 5e-4 # Factor de escala para momentos
    plt.figure(figsize=(12, 15))
    opsv.section_force_diagram_3d('Mz', sfac=sfacM)
    plt.title('Diagrama de Momento Flector (Mz)', fontweight='bold')
    plt.show()

else:
    print("No se pueden generar diagramas de fuerzas debido a un error en el análisis.")